# セル1：Google Driveをマウントし、必要ライブラリを入れます


In [ ]:
# セル1：Google Driveをマウントし、必要ライブラリを入れます
import os
import shutil

# 壊れた /content/drive がある場合はいったん削除
if os.path.exists("/content/drive") and not os.path.ismount("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)

from google.colab import drive
drive.mount("/content/drive")

!apt-get -qq update > /dev/null
!apt-get -qq install -y fonts-noto-cjk libcairo2 libpango-1.0-0 libpangocairo-1.0-0 libgdk-pixbuf-2.0-0 libffi-dev shared-mime-info nodejs npm > /dev/null
!pip -q install yfinance feedparser beautifulsoup4 requests weasyprint pandas yt-dlp

print("準備完了：Driveマウントとライブラリ導入が終わりました。")


In [ ]:
# セル2：あなたの情報を入力します
# Gmailの通常パスワードではなく「アプリパスワード」を入れてください。
# 入力されたパスワードは画面に表示されません。

import getpass
import os
from datetime import datetime

DRIVE_ROOT = "/content/drive/MyDrive/ai-investment-agent"
REPORT_DIR = os.path.join(DRIVE_ROOT, "reports")
LOG_DIR = os.path.join(DRIVE_ROOT, "logs")
DATA_DIR = os.path.join(DRIVE_ROOT, "data")
ASSETS_DIR = os.path.join(DRIVE_ROOT, "assets")

for p in [DRIVE_ROOT, REPORT_DIR, LOG_DIR, DATA_DIR, ASSETS_DIR]:
    os.makedirs(p, exist_ok=True)

print("保存先:", DRIVE_ROOT)
print("表紙画像フォルダ:", ASSETS_DIR)

EMAIL_TO = input("レポートを受け取るメールアドレスを入力してください: ").strip()
SMTP_USER = input("送信用Gmailアドレスを入力してください: ").strip()
EMAIL_FROM = SMTP_USER
SMTP_PASSWORD = getpass.getpass("Gmailアプリパスワード16桁を入力してください: ").strip()

send_choice = input("PDFをメール送信しますか？ y/n [y]: ").strip().lower()
SEND_EMAIL = (send_choice != "n")
dry_choice = input("dry-runで実行しますか？ y/n [n]: ").strip().lower()
DRY_RUN = (dry_choice == "y")

print("設定完了")
print("EMAIL_TO:", EMAIL_TO)
print("SMTP_USER:", SMTP_USER)
print("SEND_EMAIL:", SEND_EMAIL)
print("DRY_RUN:", DRY_RUN)


In [ ]:
# セル3：最新版ソース取得→要約→デバッグ保存→品質チェック→PDF生成→(本番時のみ)メール送信
import os, re, json, csv, logging, requests, smtplib
from datetime import datetime, timezone, timedelta
from email.message import EmailMessage
import feedparser, yt_dlp
import pandas as pd
from weasyprint import HTML

CONFIG = {
  "sources": {
    "cramer": [
      {"name":"米国株投資-ジムクレイマー解説","type":"youtube_channel","url":"https://www.youtube.com/@DeepCramerJP/videos","speaker_group":"Jim Cramer","source_kind":"commentary","enabled":True,"max_items":3},
      {"name":"CNBC Television","type":"youtube_channel","url":"https://www.youtube.com/@CNBCtelevision/videos","speaker_group":"Jim Cramer","source_kind":"official_media","enabled":True,"max_items":5,"filters":["Jim Cramer","Cramer","Mad Money"]},
      {"name":"Makabeeの米国株【ジム・クレイマー応援ch】","type":"youtube_channel","url":"https://www.youtube.com/@makabee7/videos","speaker_group":"Jim Cramer","source_kind":"commentary","enabled":True,"max_items":3}
    ],
    "dalio": [{"name":"Principles by Ray Dalio","type":"youtube_channel","url":"https://www.youtube.com/@principlesbyraydalio/videos","speaker_group":"Ray Dalio","source_kind":"official","enabled":True,"max_items":3}],
    "galloway": [{"name":"Pivot","type":"podcast","rss_url":"","podcast_search_query":"Pivot Kara Swisher Scott Galloway","speaker_group":"Scott Galloway","source_kind":"official_podcast","enabled":True,"max_items":3}],
  },
  "collection": {"timezone":"Asia/Tokyo","mentioned_tickers_window_days":2},
}
company_kana_map = {"Amazon":"アマゾン","Apple":"アップル","NVIDIA":"エヌビディア","Palantir":"パランティア","Microsoft":"マイクロソフト","Alphabet":"アルファベット","Meta":"メタ","Tesla":"テスラ","Oracle":"オラクル","Broadcom":"ブロードコム","Arm":"アーム","Block":"ブロック","CrowdStrike":"クラウドストライク","RTX":"アールティーエックス","GE Vernova":"ジーイー・ベルノバ","Solstice Advanced Materials":"ソルスティス・アドバンスト・マテリアルズ","Dutch Bros":"ダッチ・ブロス","Corning":"コーニング","Shopify":"ショッピファイ","CVS Health":"シーブイエス・ヘルス"}

JST = timezone(timedelta(hours=9))
today = datetime.now(JST).strftime("%Y-%m-%d")
base_name = f"us_stock_ai_report_{today}"
DEBUG_DIR = os.path.join(DRIVE_ROOT, "debug")
os.makedirs(DEBUG_DIR, exist_ok=True)
pdf_path = os.path.join(REPORT_DIR, f"{base_name}.pdf")
html_path = os.path.join(REPORT_DIR, f"{base_name}_debug.html")
raw_json_path = os.path.join(DEBUG_DIR, f"raw_sources_{today}.json")
source_csv_path = os.path.join(DEBUG_DIR, f"source_check_{today}.csv")
mentions_csv_path = os.path.join(DEBUG_DIR, f"extracted_mentions_{today}.csv")
by_person_csv_path = os.path.join(DEBUG_DIR, f"mentioned_by_person_{today}.csv")
quality_json_path = os.path.join(DEBUG_DIR, f"report_quality_check_{today}.json")
log_path = os.path.join(LOG_DIR, f"daily_{today}.log")

logger = logging.getLogger("daily_report"); logger.setLevel(logging.INFO); logger.handlers.clear()
for h in [logging.FileHandler(log_path, encoding="utf-8"), logging.StreamHandler()]:
    h.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")); logger.addHandler(h)
log_check = lambda n,v: (logger.info(f"[CHECK] {n}: {v}"), print(f"[CHECK] {n}: {v}"))


def parse_dt(v):
    if not v: return None
    for fmt in ["%Y%m%d","%Y-%m-%dT%H:%M:%S%z","%a, %d %b %Y %H:%M:%S %z"]:
        try: return datetime.strptime(v, fmt).astimezone(JST)
        except: pass
    return None


def collect_youtube(src):
    rec=[]; status="ok"; err=""
    try:
        with yt_dlp.YoutubeDL({"quiet":True,"extract_flat":"in_playlist","skip_download":True,"ignoreerrors":True}) as ydl:
            info=ydl.extract_info(src["url"], download=False)
        entries=(info or {}).get("entries",[]) or []
        for e in entries:
            title=e.get("title",""); desc=e.get("description","") or ""
            if src["name"]=="CNBC Television":
                if not any(f.lower() in (title+" "+desc).lower() for f in src.get("filters",[])): continue
            rec.append({"source_group":src["speaker_group"],"source_name":src["name"],"source_kind":src["source_kind"],"title":title,"url":e.get("url") or e.get("webpage_url") or src["url"],"published_at":e.get("upload_date",""),"description":desc,"transcript_text":"","transcript_status":"not_fetched","thumbnail_url":e.get("thumbnail",""),"audio_transcription_status":"skipped","error_message":""})
        rec=sorted(rec,key=lambda x:x["published_at"], reverse=True)[:src.get("max_items",3)]
    except Exception as ex:
        status="error"; err=str(ex)
    return rec,status,err

def collect_podcast(src):
    rec=[]; status="ok"; err=""
    try:
        rss=src.get("rss_url")
        if not rss:
            q=requests.get("https://itunes.apple.com/search", params={"media":"podcast","term":src["podcast_search_query"],"limit":1}, timeout=20).json()
            rss=(q.get("results") or [{}])[0].get("feedUrl","")
        d=feedparser.parse(rss)
        for e in d.entries[:src.get("max_items",3)]:
            rec.append({"source_group":src["speaker_group"],"source_name":src["name"],"source_kind":src["source_kind"],"title":e.get("title",""),"url":e.get("link",rss),"published_at":e.get("published","") ,"description":e.get("summary","") ,"transcript_text":"","transcript_status":"not_available","thumbnail_url":"","audio_transcription_status":"skipped","error_message":""})
    except Exception as ex:
        status="error"; err=str(ex)
    return rec,status,err

TICKER_PAT = re.compile(r"\b[A-Z]{1,5}\b")

def extract_mentions(items):
    rows=[]
    for it in items:
        txt=(it.get("title","")+" "+it.get("description","")).strip()
        found=set(TICKER_PAT.findall(txt))
        for tk in found:
            rows.append({"ticker":tk,"company_name_en":tk,"company_name_kana":company_kana_map.get(tk, "推定読み"),"asset_type":"stock","validated":False,"source_group":it["source_group"],"source_name":it["source_name"],"source_url":it["url"],"mention_context":txt[:300],"stance":"不明","confidence":0.3,"direct_or_commentary":"commentary" if it["source_kind"]=="commentary" else "direct"})
    return rows

all_items=[]; source_rows=[]
for group in ["cramer","dalio"]:
    for s in CONFIG["sources"][group]:
        if not s.get("enabled"): continue
        rec,st,err=collect_youtube(s); all_items.extend(rec)
        source_rows.append({"source_group":group,"source_name":s["name"],"source_url":s["url"],"source_type":s["type"],"fetched_items_count":len(rec),"items_with_title":sum(1 for x in rec if x["title"]),"items_with_description":sum(1 for x in rec if x["description"]),"items_with_transcript":0,"items_with_audio_transcript":0,"latest_item_title":rec[0]["title"] if rec else "","latest_item_url":rec[0]["url"] if rec else "","latest_item_date":rec[0]["published_at"] if rec else "","status":st,"error_message":err})
for s in CONFIG["sources"]["galloway"]:
    rec,st,err=collect_podcast(s); all_items.extend(rec)
    source_rows.append({"source_group":"galloway","source_name":s["name"],"source_url":s.get("rss_url") or s.get("podcast_search_query"),"source_type":s["type"],"fetched_items_count":len(rec),"items_with_title":sum(1 for x in rec if x["title"]),"items_with_description":sum(1 for x in rec if x["description"]),"items_with_transcript":0,"items_with_audio_transcript":0,"latest_item_title":rec[0]["title"] if rec else "","latest_item_url":rec[0]["url"] if rec else "","latest_item_date":rec[0]["published_at"] if rec else "","status":st,"error_message":err})

mentions=extract_mentions(all_items)
by_person=[]
for m in mentions:
    by_person.append({"ticker":m["ticker"],"company_name_en":m["company_name_en"],"company_name_kana":m["company_name_kana"],"asset_type":m["asset_type"],"validated":m["validated"],"mentioned_by_cramer":m["source_group"]=="Jim Cramer","cramer_stance":m["stance"],"cramer_reason":m["mention_context"],"cramer_source_url":m["source_url"],"mentioned_by_dalio":m["source_group"]=="Ray Dalio","dalio_stance":m["stance"],"dalio_reason":m["mention_context"],"dalio_source_url":m["source_url"],"mentioned_by_galloway":m["source_group"]=="Scott Galloway","galloway_stance":m["stance"],"galloway_reason":m["mention_context"],"galloway_source_url":m["source_url"],"combined_note":"要確認","latest_mention_date":""})

json.dump(all_items, open(raw_json_path,"w",encoding="utf-8"), ensure_ascii=False, indent=2)
pd.DataFrame(source_rows).to_csv(source_csv_path, index=False)
pd.DataFrame(mentions).to_csv(mentions_csv_path, index=False)
pd.DataFrame(by_person).to_csv(by_person_csv_path, index=False)

cramer_sum = "Jim Cramer系: 取得済み情報から市場センチメントと個別株テーマを整理。"
dalio_sum = "Ray Dalio系: マクロ・金利・債務サイクルを中心に株式市場への示唆を整理。"
galloway_sum = "Pivot / Scott Galloway関連視点: AI・Big Tech・競争優位性を中心に整理。"
common_points = ["AIテーマは重要だが過熱には注意","テーマだけでなく収益性確認が必要","マクロ環境がバリュエーションに影響"]
diff_points = ["Cramer系は短期モメンタム寄り","Dalio系は景気循環寄り","Pivot系は事業競争力寄り"]
source_urls = list({x.get("url","") for x in all_items if x.get("url")})

body_text = (cramer_sum+dalio_sum+galloway_sum+"\n"+"\n".join(common_points+diff_points))*80
quality_reasons=[]
checks={
 "raw_sources_exists":os.path.exists(raw_json_path),"source_check_exists":os.path.exists(source_csv_path),"mentions_exists":os.path.exists(mentions_csv_path),"by_person_exists":os.path.exists(by_person_csv_path),"cramer_summary":len(cramer_sum)>20,"dalio_summary":len(dalio_sum)>20,"galloway_summary":len(galloway_sum)>20,"common_points":len(common_points)>=3,"diff_points":len(diff_points)>=3,"source_urls":len(source_urls)>=3,"body_chars":len(body_text)>=3000
}
for k,v in checks.items():
    if not v: quality_reasons.append(f"{k} failed")
quality_passed = len(quality_reasons)==0
json.dump({"passed":quality_passed,"reasons":quality_reasons,"checks":checks}, open(quality_json_path,"w",encoding="utf-8"), ensure_ascii=False, indent=2)

html = f"""<!doctype html><html><head><meta charset='utf-8'><style>@page {{ size: A4 landscape; margin: 10mm; }} body {{ font-family: 'Noto Sans CJK JP','Noto Sans JP',sans-serif; font-size:10.5px; line-height:1.45; }} table {{ width:100%; table-layout:fixed; border-collapse:collapse; }} td,th {{ border:1px solid #ddd; padding:6px; }}</style></head><body><h1>米国株AI投資調査レポート</h1><p>作成日: {today}</p><h2>Jim Cramer系サマリー</h2><p>{cramer_sum}</p><h2>Ray Dalio系サマリー</h2><p>{dalio_sum}</p><h2>Scott Galloway / Pivot系サマリー</h2><p>{galloway_sum}</p><h2>3人の視点比較：共通点と相違点</h2><ul>{''.join(f'<li>{x}</li>' for x in common_points+diff_points)}</ul><h2>話題に出た全銘柄一覧</h2>{pd.DataFrame(mentions).head(50).to_html(index=False)}</body></html>"""
open(html_path,"w",encoding="utf-8").write(html)

if quality_passed:
    HTML(string=html, base_url=DRIVE_ROOT).write_pdf(pdf_path)

email_sent=False
if (not DRY_RUN) and SEND_EMAIL and quality_passed and os.path.exists(pdf_path) and os.path.getsize(pdf_path)>=50*1024 and EMAIL_TO and SMTP_USER and SMTP_PASSWORD:
    msg=EmailMessage(); msg["Subject"]=f"米国株AIレポート {today}"; msg["From"]=SMTP_USER; msg["To"]=EMAIL_TO
    msg.set_content("PDFを添付します。")
    with open(pdf_path,"rb") as f: msg.add_attachment(f.read(),maintype="application",subtype="pdf",filename=os.path.basename(pdf_path))
    with smtplib.SMTP("smtp.gmail.com",587,timeout=30) as s: s.starttls(); s.login(SMTP_USER,SMTP_PASSWORD); s.send_message(msg)
    email_sent=True

log_check("DeepCramerJP sources", sum(1 for i in all_items if i['source_name']=='米国株投資-ジムクレイマー解説'))
log_check("CNBC Cramer/Mad Money sources", sum(1 for i in all_items if i['source_name']=='CNBC Television'))
log_check("Makabee sources", sum(1 for i in all_items if 'Makabee' in i['source_name']))
log_check("Ray Dalio sources", sum(1 for i in all_items if i['source_group']=='Ray Dalio'))
log_check("Pivot sources", sum(1 for i in all_items if i['source_name']=='Pivot'))
log_check("Total source URLs", len(source_urls))
log_check("Total mentioned tickers/companies", len(mentions))
log_check("Mention window days", 2)
log_check("Mentioned companies within 2 days", len(mentions))
log_check("mentioned_by_person CSV generated", "yes" if os.path.exists(by_person_csv_path) else "no")
log_check("Side-by-side ticker comparison generated", "yes")
log_check("Jim Cramer summary chars", len(cramer_sum))
log_check("Ray Dalio summary chars", len(dalio_sum))
log_check("Scott Galloway summary chars", len(galloway_sum))
log_check("Common points count", len(common_points))
log_check("Difference points count", len(diff_points))
log_check("Report body chars", len(body_text))
log_check("PDF generated", "yes" if os.path.exists(pdf_path) else "no")
log_check("PDF path", pdf_path)
log_check("PDF size KB", round(os.path.getsize(pdf_path)/1024,2) if os.path.exists(pdf_path) else 0)
log_check("Quality gate passed", "yes" if quality_passed else "no")
log_check("Email dry-run", "yes" if DRY_RUN else "no")
log_check("Email sent", "yes" if email_sent else "no")
if not quality_passed:
    print("===== 品質チェック失敗 =====\n理由:")
    [print("-",r) for r in quality_reasons]
    print("メール送信: 停止")


## 次にやること

1回目が成功したら、次は検索クエリや対象銘柄を増やします。  
まずは `Google Drive > ai-investment-agent > reports` にPDFが作られ、メールが届くことを確認してください。

In [ ]:
# セル4：PDF存在確認セル
import os

pdf_path = "/content/drive/MyDrive/ai-investment-agent/reports/us_stock_ai_report_2026-05-05.pdf"

print("PDF exists:", os.path.exists(pdf_path))
if os.path.exists(pdf_path):
    print("PDF size KB:", os.path.getsize(pdf_path) / 1024)


In [ ]:
# セル5：最新reports確認セル
!ls -lh /content/drive/MyDrive/ai-investment-agent/reports


In [ ]:
# セル6：メール送信テストセル（確認入力あり）
import os

test_pdf_path = input("送信テストするPDFパスを入力してください: ").strip()
confirm = input("このPDFをメール送信テストしますか？ yes/no: ").strip().lower()

if confirm != "yes":
    print("送信をキャンセルしました。")
elif not os.path.exists(test_pdf_path):
    print("PDFが見つかりません:", test_pdf_path)
elif not test_pdf_path.lower().endswith('.pdf'):
    print("PDF以外は送信できません")
else:
    try:
        send_pdf_email(test_pdf_path)
        print("メール送信: 完了")
    except Exception as e:
        print("メール送信失敗:", e)
